# A Handbag-Shoes classifier using CNN and ImageNet 

## Initialization

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras

keras.utils.set_random_seed(42)

2026-01-18 16:29:28.418714: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-18 16:29:28.456623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-18 16:29:29.396312: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Getting the Data

In [2]:


def get_data():
    
    import os
    import subprocess
    from pathlib import Path

    DATA_DIR = Path("./handbags-shoes")
    ZIP_FILE = Path("./handbags-shoes.zip")
    URL = "https://www.dropbox.com/s/w07liww46kgxo1m/handbags-shoes.zip?dl=1"

    # Case 1: data already extracted
    if DATA_DIR.exists():
        print(f"✅ Data already present at: {DATA_DIR.resolve()}")
        return DATA_DIR

    # Case 2: zip exists but not extracted
    if ZIP_FILE.exists():
        print("📦 Zip file found. Extracting...")
        subprocess.run(
            ["unzip", "-qq", str(ZIP_FILE)],
            check=True
        )
        return DATA_DIR

    # Case 3: nothing exists → download
    print("⬇️ Downloading dataset...")
    subprocess.run(
        ["wget", "-O", str(ZIP_FILE), URL],
        check=True
    )

    print("📦 Extracting dataset...")
    subprocess.run(
        ["unzip", "-qq", str(ZIP_FILE)],
        check=True
    )

    return DATA_DIR
data_path = get_data()
print(data_path)


✅ Data already present at: /home/saurabh-singh/Documents/GitHub/MastersDS/MIT-15.773/handbags-shoes
handbags-shoes


## Data Pre-Processing

In [3]:
import os
import shutil
import pathlib

base_dir = pathlib.Path.cwd() / "handbags-shoes"

splits = {
    "train": slice(0, 50),
    "validation": slice(50, 75),
    "test": slice(75, None),
}

for category in ("handbags", "shoes"):
    fnames = os.listdir(base_dir / category)

    for split, sl in splits.items():
        out_dir = base_dir / split / category
        os.makedirs(out_dir, exist_ok=True)

        for fname in fnames[sl]:
            src = base_dir / category / fname
            dst = out_dir / fname

            # avoid re-copying if the file already exists
            if not dst.exists():
                shutil.copyfile(src, dst)


In [4]:
# Loading the data into Kears 

train_dataset = keras.utils.image_dataset_from_directory(
    base_dir / "train" , 
    image_size = (224, 224),
    batch_size = 32,
)

validation_dataset = keras.utils.image_dataset_from_directory(
    base_dir / "validation",
    image_size = (224, 224), 
    batch_size = 32,
)
validation_dataset = keras.utils.image_dataset_from_directory(
    base_dir / "validation",
    image_size = (224, 224), 
    batch_size = 32,
)
test_dataset = keras.utils.image_dataset_from_directory(
    base_dir / "test",
    image_size = (224, 224), 
    batch_size = 32,
)

Found 97 files belonging to 2 classes.


I0000 00:00:1768733977.676451  149439 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10247 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


Found 50 files belonging to 2 classes.
Found 50 files belonging to 2 classes.
Found 38 files belonging to 2 classes.


In [ ]:
for images, _ in train_dataset.take(1):
    print(images[0].shape)

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1 )
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.axis("off")

## CNN Model 

In [ ]:
input = keras.Input(shape=(224,224,3)) # in torch it should be (3,244,244)

h = keras.layers.Rescaling(1./255)(input)

h = keras.layers.Conv2D(
    32,
    kernel_size = (2,2),
    activation="relu",
    name="Conv_1"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.Conv2D(
    32,
    kernel_size = (2,2),
    activation="relu",
    name="Conv_2"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.Flatten()(h)

output = keras.layers.Dense(1, activation="sigmoid")(h)
model = keras.Model(input, output)
model.summary()

In [5]:
# Standard function to print loss 

def plot_loss_curves(history):
    plt.clf()
    history_dict = history.history
    loss_values = history_dict["loss"]
    validation_loss_values = history_dict["val_loss"]
    epochs = range(1, len(loss_values) + 1)
    plt.plot(epochs, loss_values, "bo", label="Training Loss")
    plt.plot(epochs, validation_loss_values, "b", label="Validation Loss")
    plt.title("Training and validation loss")
    plt.xlabel("epochs")
    plt.ylabel("loss")
    plt.legend()
    plt.show()
    
def plot_acc_curves(history):
    plt.clf()
    history_dict = history.history
    acc = history_dict["accuracy"]
    validation_acc = history_dict["val_accuracy"]
    epochs = range(1, len(acc) + 1)
    plt.plot(epochs, acc, "bo", label="Training accuracy")
    plt.plot(epochs, validation_acc, "b", label="validation accuracy")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()
    
    

## Training CNN Model 

In [ ]:
model.compile(
    loss = "binary_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data = validation_dataset
)

plot_acc_curves(history=history)
plot_loss_curves(history=history)



In [ ]:
score = model.evaluate(test_dataset)
print("test accuracy : ", score[1])

## Data Augmentation

In [ ]:
def augment_data(image):
    x = keras.layers.RandomFlip("horizontal")(image)
    x = keras.layers.RandomRotation(0.1)(x)
    x = keras.layers.RandomZoom(0.2)(x)
    return x

In [ ]:
plt.figure(figsize=(10,10))
for images, _ in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = augment_data(images[0])
        plt.imshow(augmented_image.numpy().astype("uint8"))
        plt.axis("off")


## Model with augmented data

In [ ]:
input = keras.Input(shape=(224,224,3)) # in torch it should be (3,244,244)

# ------ Augmented Layer in the previous model --------------------

h = keras.layers.RandomFlip("horizontal")(input)
h = keras.layers.RandomRotation(0.1)(h)
h = keras.layers.RandomZoom(0.2)(h)


# ================================================================

h = keras.layers.Rescaling(1./255)(h)

h = keras.layers.Conv2D(
    32,
    kernel_size = (2,2),
    activation="relu",
    name="Conv_1"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.Conv2D(
    32,
    kernel_size = (2,2),
    activation="relu",
    name="Conv_2"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.Flatten()(h)

output = keras.layers.Dense(1, activation="sigmoid")(h)
model = keras.Model(input, output)
model.summary()

In [ ]:
model.compile(
    loss = "binary_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]
)

In [ ]:

history = model.fit(
    train_dataset,
    epochs=80,
    validation_data = validation_dataset
)

plot_acc_curves(history=history)
plot_loss_curves(history=history)
score = model.evaluate(test_dataset)
print("test accuracy : ", score[1])

## Model 3 with changes from chatgpt

In [ ]:
input = keras.Input(shape=(224,224,3)) # in torch it should be (3,244,244)

# ------ Augmented Layer in the previous model --------------------

h = keras.layers.RandomFlip("horizontal")(input)
h = keras.layers.RandomRotation(0.05)(h)
h = keras.layers.RandomZoom(0.1)(h)


# ================================================================

h = keras.layers.Rescaling(1./255)(h)

h = keras.layers.Conv2D(
    32,
    kernel_size = (3,3),
    activation="relu",
    name="Conv_1"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.Conv2D(
    32,
    kernel_size = (3,3),
    activation="relu",
    name="Conv_2"
)(h)

h = keras.layers.MaxPool2D()(h)

h = keras.layers.GlobalAveragePooling2D()(h)

output = keras.layers.Dense(1, activation="sigmoid")(h)
model = keras.Model(input, output)
model.summary()

model.compile(
    loss = "binary_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]
)

In [ ]:

history = model.fit(
    train_dataset,
    epochs=20,
    validation_data = validation_dataset
)

plot_acc_curves(history=history)
plot_loss_curves(history=history)
score = model.evaluate(test_dataset)
print("test accuracy : ", score[1])

## Usin Resnet50 for transfer learining

In [ ]:
resnet50_base = keras.applications.ResNet50(
    weights="imagenet",
    include_top = False,
    input_shape=(224,224,3)
)

resnet50_base.summary()

In [ ]:
# getting features and labels from the resnet

def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        preprocessed_images = keras.applications.resnet50.preprocess_input(images)
        features = resnet50_base.predict(preprocessed_images)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features , train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

In [ ]:
train_features.shape

In [ ]:
# transfer learning 

input = keras.layers.Input(shape=(7,7,2048))
h = keras.layers.Flatten()(input)
h = keras.layers.Dense(256, activation="relu")(h)
h = keras.layers.Dropout(0.5)(h)
output = keras.layers.Dense(1,activation="sigmoid")(h)
model = keras.Model(input,output)
model.summary()

In [ ]:
model.compile(
    loss = "binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


In [ ]:

history = model.fit(
    train_features,
    train_labels,
    epochs=10,
    validation_data = (val_features, val_labels)
)

plot_acc_curves(history=history)
plot_loss_curves(history=history)
score = model.evaluate(test_features,test_labels)
print("test accuracy : ", score[1])

## Using Huggingface

### To be done later

In [5]:
from transformers import AutoImageProcessor, TFAutoModel

In [ ]:
def get_vit_features_and_labels(dataset):
    all_features = []
    all_labels = []

    for images, labels in dataset:
        images = tf.cast(images, tf.float32)

        inputs = processor(images=images, return_tensors="tf")
        outputs = vit_base(**inputs, training=False)

        # CLS token → global image representation
        features = outputs.last_hidden_state[:, 0, :]

        all_features.append(features.numpy())
        all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)


In [6]:
checkpoint = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(checkpoint)
vit_base = TFAutoModel.from_pretrained(checkpoint)
vit_base.trainable(False)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


TypeError: 'builtins.safe_open' object is not iterable